# SST DIO: Minian parameter review, stage by stage (G27 HC2)

Plan: `plans/sst_dio_dreadd_validation_plan.md` §6. The pipeline is walked one stage at a time, as in the Minian notebook: each stage's variants are judged **at that stage's own output**, decided, and the decision is carried into the next stage's grid. Claude's recommendation for each stage is in the plan, in bold. Kernel: `minian-native`.

Stages: 1 preprocessing (denoise, background removal) · 2 seeds · 3 initialisation · 4 CNMF updates and merges. At the end: the earlier one-at-a-time grid, kept as a sensitivity map.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.expanduser('~/code/caban_sst'))
sys.path.insert(0, os.path.expanduser('~/code/minian_vsekulic'))

import holoviews as hv
import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Image
from minian.visualization import CNMFViewer, VArrayViewer

from caban import sst_dio_parameter_review as review

%matplotlib inline
hv.notebook_extension('bokeh', width=100)
output_size = 100
STAGES_ROOT = os.path.expanduser('~/sst_dio_out/parameter_stages/G27_HC2')
GRID_ROOT = os.path.expanduser('~/sst_dio_out/parameter_grid/G27_HC2')

## Stage 1: preprocessing (denoise `ksize` × background `wnd`)

Measurements on the same 52 reference cells for every variant: `contrast_z` (cell vs ring in the max projection, in background s.d.), `transient_snr` (cell-minus-ring trace, peak over noise), `residual_background` (large-scale structure left away from cells, relative to cell contrast), `core_fwhm_px` (cell width; a window smaller than the cell narrows it).

In [ ]:
STAGE1 = os.path.join(STAGES_ROOT, 'preprocessing')
display(pd.read_csv(os.path.join(STAGE1, 'stage_metrics.csv'), index_col=0).round(3))
display(Image(os.path.join(STAGE1, 'stage_metrics.png')))
display(Image(os.path.join(STAGE1, 'max_proj_grid.png')))

### Any variant against ksize 5 / wnd 15 (the current values)

In [ ]:
@widgets.interact(ksize=widgets.Dropdown(options=[3, 5, 7], value=5),
                  wnd=widgets.Dropdown(options=[10, 15, 20, 25, 30], value=20))
def show_preprocessing(ksize, wnd):
    review.compare_preprocessing(STAGE1, 'ksize_{}_wnd_{}'.format(ksize, wnd))
    plt.show()

### The movies, side by side (Minian's viewer)

Set the variant, run the cell. Left: the current values; right: the variant (motion-corrected, after background removal).

In [ ]:
VARIANT = 'ksize_5_wnd_20'

Y_ref = review.open_stage_array(STAGE1, 'ksize_5_wnd_15', 'Y_fm_chk', intermediate=True)
Y_var = review.open_stage_array(STAGE1, VARIANT, 'Y_fm_chk', intermediate=True)
hv.output(size=int(output_size * 0.6))
vaviewer = VArrayViewer([Y_ref.rename('ksize_5_wnd_15'), Y_var.rename(VARIANT)],
                        framerate=5, summary=None, layout=True)
display(vaviewer.show())

In [ ]:
# review.log_decision(STAGES_ROOT, stage='preprocessing', values_tried='ksize 3/5/7 x wnd 10-30',
#                     value_chosen='ksize 5, wnd 15', why='...')

## Stages 2-4

Built once stage 1 is decided.

## Sensitivity map: the one-at-a-time grid (full runs, final units)

Each variant changed one value from the current set and ran the whole pipeline; only the final units are shown, so a difference mixes the changed stage with everything after it. Kept for which parameters matter at all.

In [ ]:
display(review.summary_table(GRID_ROOT))
finished = [name for name, r in review.variants(GRID_ROOT).items()
            if r['status'] == 'done' and name != 'frozen']


@widgets.interact(variant=widgets.Dropdown(options=finished))
def show(variant):
    review.compare(GRID_ROOT, variant)
    plt.show()

In [ ]:
VARIANT = 'frozen'

A = review.open_array(GRID_ROOT, VARIANT, 'A')
C = review.open_array(GRID_ROOT, VARIANT, 'C')
S = review.open_array(GRID_ROOT, VARIANT, 'S')
Y_fm_chk = review.open_array(GRID_ROOT, VARIANT, 'Y_fm_chk', intermediate=True)
cnmfviewer = CNMFViewer(A=A, C=C, S=S, org=Y_fm_chk)
hv.output(size=int(output_size * 0.7))
display(cnmfviewer.show())